In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision transformers -q
# !pip install matplotlib numpy Pillow requests -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_install("torch")
_install("torchvision")
_install("transformers")
_install("matplotlib")
_install("numpy")
_install("Pillow")
_install("requests")

In [ ]:
import math
import os

import requests
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ViLD 复试讲解版：5 分钟讲清 Open-Vocabulary Detection

基于论文 *Open-vocabulary Object Detection via Vision and Language Knowledge Distillation*（Gu et al., 2022, ICLR），
这一版 notebook 不追求“讲全”，而追求“讲清”。

你只需要抓住 3 句话：

1. **传统检测器只能识别训练时见过的类别**，因为分类头是固定的。
2. **ViLD-text** 用文本 embedding 替代固定分类器，让 detector 可以和任意类别文本做相似度分类。
3. **ViLD-image** 再用 CLIP 图像编码器做蒸馏，把开放世界视觉语义教给 detector，所以它不只会 base classes。

> 这一版 notebook 的结构只保留复试最有用的主线：
> - 问题是什么
> - 方法怎么做
> - 训练和推理有什么区别
> - 一个可运行 demo
> - 最后怎么答面试题

## 1. 先记住 ViLD 在解决什么问题

传统 Faster R-CNN / Mask R-CNN 的分类头本质上是一个固定大小的线性分类器。
这意味着：

- 训练时定义了多少类，推理时就只能认多少类；
- 就算 proposal 很准，如果类别没在训练标签里出现，模型也说不出来。

ViLD 要解决的就是这个问题：

> **训练时只学基础类（base classes），推理时仍然能检测新类别（novel classes）。**

所以你可以把 ViLD 看成：
**“把 closed-set detector 改造成 text-driven detector，并用 CLIP 把开放世界语义蒸馏进去。”**

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

print(f"CLIP loaded on {device}. projection_dim={clip_model.config.projection_dim}")

In [ ]:
# ── 只保留一张图，后面统一用于 ViLD / OWL-ViT 演示 ──
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(IMAGE_URL, stream=True).raw).convert("RGB")

plt.figure(figsize=(5, 4))
plt.imshow(image)
plt.title("Demo Image")
plt.axis("off")
plt.show()

print(f"Image size: {image.size}")

## 2. ViLD 的方法主线

复试里最推荐的讲法不是按论文顺序逐段背，而是按下面 3 步讲：

### Step 1：传统 detector 为什么不行？
因为它的分类头是固定的，只能输出训练时定义好的类别。

### Step 2：ViLD-text 做了什么？
把分类头改成：

$$
\text{score} = \text{region embedding} \cdot \text{text embedding}^{\top}
$$

这样类别不再写死在参数里，而是由文本决定。

### Step 3：为什么这样还不够？
因为只靠 base classes 的检测监督，detector 学到的视觉语义还是有限。
所以 ViLD 又加了 `ViLD-image` 分支，用 CLIP image encoder 做蒸馏，把开放世界语义教给 detector。

In [ ]:
# ── 传统 detector 的核心限制：分类头维度固定 ──
ROI_FEAT_DIM = 256 * 7 * 7
HIDDEN_DIM = 1024
NUM_BASE_CLASSES = 80

roi_head = nn.Sequential(
    nn.Linear(ROI_FEAT_DIM, HIDDEN_DIM),
    nn.ReLU(),
    nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
    nn.ReLU(),
)
fixed_classifier = nn.Linear(HIDDEN_DIM, NUM_BASE_CLASSES + 1)  # +1 for background

sample_roi = torch.randn(4, ROI_FEAT_DIM)
region_feat = roi_head(sample_roi)                               # (4, 1024)
cls_logits = fixed_classifier(region_feat)                       # (4, 81)

print(f"Region feature shape: {tuple(region_feat.shape)}")
print(f"Classification logits shape: {tuple(cls_logits.shape)}")
print(f"Classifier weight shape: {tuple(fixed_classifier.weight.shape)}")
print("结论：类别维度在训练时就固定了，所以传统 detector 天生是 closed-set。")

### 2.1 ViLD-text：把分类器换成文本原型

ViLD-text 的一句话理解：

> 不再学习一个固定类别分类器，而是把类别名称送进 CLIP text encoder，生成文本原型。

公式上最重要的是：

$$
\text{score} = \mathbf{e}_R [\mathbf{e}_T; \mathbf{e}_{bg}]^\top
$$

其中：
- $\mathbf{e}_R$：proposal 的 region embedding
- $\mathbf{e}_T$：基础类的文本 embedding
- $\mathbf{e}_{bg}$：可学习的背景 embedding

这一步解决的是：**怎么把 closed-set 分类器改成 open-vocabulary 分类器。**

In [ ]:
# ── 模拟 ViLD-text 分支 ──

class ViLDTextBranch(nn.Module):
    """RoI feature -> projection -> region embedding -> text similarity"""

    def __init__(self, roi_feat_dim=256 * 7 * 7, hidden_dim=1024, embed_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(roi_feat_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.projection = nn.Linear(hidden_dim, embed_dim)    # (N, 1024) -> (N, 512)
        self.bg_embedding = nn.Parameter(torch.randn(1, embed_dim))

    def get_region_embeddings(self, roi_features):
        x = F.relu(self.fc1(roi_features))                     # (N, roi_feat_dim) -> (N, 1024)
        x = F.relu(self.fc2(x))                                # (N, 1024) -> (N, 1024)
        x = self.projection(x)                                 # (N, 1024) -> (N, 512)
        x = F.normalize(x, p=2, dim=-1)                        # (N, 512)
        return x

    def forward(self, roi_features, text_embeddings):
        region_embeds = self.get_region_embeddings(roi_features)           # (N, 512)
        bg_norm = F.normalize(self.bg_embedding, p=2, dim=-1)              # (1, 512)
        classifier = torch.cat([text_embeddings, bg_norm], dim=0)          # (C_B+1, 512)
        logits = region_embeds @ classifier.T                              # (N, C_B+1)
        return logits, region_embeds

base_categories = ["cat", "dog", "remote control", "couch", "person"]
prompts = [f"a photo of a {name}" for name in base_categories]
text_inputs = clip_processor(text=prompts, return_tensors="pt", padding=True)
text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
with torch.no_grad():
    text_embeds = clip_model.get_text_features(**text_inputs)              # (5, 512)
    text_embeds = F.normalize(text_embeds, p=2, dim=-1)

N_PROPOSALS = 10
ROI_FEAT_DIM = 256 * 7 * 7
vild_text = ViLDTextBranch(roi_feat_dim=ROI_FEAT_DIM).to(device)
dummy_roi = torch.randn(N_PROPOSALS, ROI_FEAT_DIM, device=device)
logits, region_embeds = vild_text(dummy_roi, text_embeds)
all_cats = base_categories + ["background"]

print("Base categories:", base_categories)
print("Text embeddings shape:", tuple(text_embeds.shape))
print("Region embeddings shape:", tuple(region_embeds.shape))
print("Similarity logits shape:", tuple(logits.shape))
print("Sample prediction logits (proposal 0):")
for category, score in zip(all_cats, logits[0].detach().cpu().tolist()):
    print(f"  {category:<16} -> {score:.4f}")

### 2.2 ViLD-image：把 CLIP 的视觉语义蒸馏给 detector

如果只有 ViLD-text，模型虽然已经能“用文本分类”，但它学到的 detector 特征仍主要来自 base classes。

所以 ViLD 再引入一个 teacher-student 蒸馏分支：

- **Teacher**：冻结的 CLIP image encoder
- **Student**：detector 的 RoI head + projection

蒸馏目标很直接：

$$
\mathcal{L}_{image} = \frac{1}{M}\sum_{i=1}^{M}\|\mathbf{e}_{R_i} - \mathbf{e}_{I_i}\|_1
$$

这一步解决的是：**怎么把 CLIP 的开放世界视觉语义注入 detector。**

In [ ]:
# ── 模拟 ViLD-image 蒸馏流程 ──

def simulate_vild_image_distillation(image, proposals_boxes, clip_model, clip_processor, student_head):
    teacher_embeds = []
    cropped_regions = []
    width, height = image.size
    num_boxes = len(proposals_boxes)

    for box in proposals_boxes:
        x1, y1, x2, y2 = box
        crop = image.crop((int(x1 * width), int(y1 * height), int(x2 * width), int(y2 * height)))
        cropped_regions.append(crop)

        teacher_inputs = clip_processor(images=crop, return_tensors="pt")
        teacher_inputs = {k: v.to(device) for k, v in teacher_inputs.items()}
        with torch.no_grad():
            teacher_embed = clip_model.get_image_features(**teacher_inputs)        # (1, 512)
            teacher_embed = F.normalize(teacher_embed, p=2, dim=-1)
        teacher_embeds.append(teacher_embed)

    teacher_embeds = torch.cat(teacher_embeds, dim=0)                             # (M, 512)

    dummy_roi = torch.randn(num_boxes, ROI_FEAT_DIM, device=device)               # (M, 12544)
    student_embeds = student_head.get_region_embeddings(dummy_roi)                 # (M, 512)
    distill_loss = F.l1_loss(student_embeds, teacher_embeds)
    return teacher_embeds, student_embeds, distill_loss, cropped_regions

proposals_boxes = [
    (0.00, 0.30, 0.45, 0.95),
    (0.35, 0.25, 0.75, 0.90),
    (0.25, 0.00, 0.50, 0.25),
    (0.60, 0.00, 0.85, 0.30),
    (0.00, 0.00, 1.00, 1.00),
]

teacher_e, student_e, loss, crops = simulate_vild_image_distillation(
    image=image,
    proposals_boxes=proposals_boxes,
    clip_model=clip_model,
    clip_processor=clip_processor,
    student_head=vild_text,
)

print("Teacher embeddings shape:", tuple(teacher_e.shape))
print("Student embeddings shape:", tuple(student_e.shape))
print("L1 distillation loss:", round(loss.item(), 4))

fig, axes = plt.subplots(1, len(crops), figsize=(15, 3))
for idx, (ax, crop) in enumerate(zip(axes, crops)):
    ax.imshow(crop)
    ax.set_title(f"Proposal {idx}")
    ax.axis("off")
fig.suptitle("Teacher Crops for ViLD-image Distillation", fontsize=12)
plt.tight_layout()
plt.show()

### 2.3 训练 vs 推理：复试最容易被问的一点

完整损失可以写成：

$$
\mathcal{L}_{total} = \mathcal{L}_{text}^{CE} + \lambda \mathcal{L}_{image}^{L1} + \mathcal{L}_{box}
$$

但真正要记住的是下面这张表：

| 阶段 | 发生了什么 |
|------|------------|
| 训练 | `ViLD-text` 做分类监督，`ViLD-image` 做 CLIP 蒸馏 |
| 推理 | 只保留 detector 输出的 `region embedding` 与 `text embedding` 做相似度分类 |

所以老师如果问：
**“为什么推理时不需要 CLIP image encoder？”**
最标准的回答就是：

> 因为它只在训练时充当 teacher，把开放世界语义蒸馏给 detector；推理时 student 已经学会了。

In [ ]:
# ── ViLD 推理公式：真实路径 ──

def build_text_embeddings(category_names, clip_model, clip_processor):
    prompts = [f"a photo of a {name}" for name in category_names]
    text_inputs = clip_processor(text=prompts, return_tensors="pt", padding=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    with torch.no_grad():
        text_embeds = clip_model.get_text_features(**text_inputs)
        text_embeds = F.normalize(text_embeds, p=2, dim=-1)
    return text_embeds


def vild_inference_from_region_embeddings(region_embeds, category_names, clip_model, clip_processor):
    text_embeds = build_text_embeddings(category_names, clip_model, clip_processor)
    scores = region_embeds @ text_embeds.T
    probs = scores.softmax(dim=-1)
    predictions = probs.argmax(dim=-1)
    return probs, predictions


NUM_DEMO_PROPOSALS = 5
with torch.no_grad():
    simulated_region_embeds = vild_text.get_region_embeddings(
        torch.randn(NUM_DEMO_PROPOSALS, ROI_FEAT_DIM, device=device)
    )

base_cats = ["cat", "dog", "remote control", "couch"]
base_probs, base_preds = vild_inference_from_region_embeddings(
    simulated_region_embeds,
    base_cats,
    clip_model,
    clip_processor,
)

print("Real ViLD inference formula:")
for idx, pred_idx in enumerate(base_preds.detach().cpu().tolist()):
    print(f"  Proposal {idx}: {base_cats[pred_idx]:>16s}  prob={base_probs[idx, pred_idx].item():.3f}")

print("\n一句话：真实 ViLD 推理时，不调用 CLIP image encoder，只做 region-text similarity。")

## 3. 一个可运行 demo：用 OWL-ViT 感受 open-vocabulary detection

ViLD 官方实现更偏研究工程，不适合在复试现场直接演示。
所以这里保留一个最实用的替代 demo：**OWL-ViT**。

你在复试里可以这样说：

> ViLD 负责帮我理解“为什么 open-vocabulary detection 可行”；
> OWL-ViT 负责帮我演示“这类模型怎么实际跑起来”。

这一节只保留同一张图片上的两组 query：
- 基础类 query
- 更细粒度的新类别 query

In [ ]:
from transformers import OwlViTForObjectDetection, OwlViTProcessor

owlvit_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owlvit_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)
owlvit_model.eval()

print(f"OWL-ViT loaded on {device}.")

In [ ]:
# ── 场景1：基础类别检测 ──
text_labels = [["a photo of a cat", "a photo of a dog", "a photo of a remote control"]]

owl_inputs = owlvit_processor(text=text_labels, images=image, return_tensors="pt")
owl_inputs = {k: v.to(device) for k, v in owl_inputs.items()}
with torch.no_grad():
    owl_outputs = owlvit_model(**owl_inputs)

target_sizes = torch.tensor([(image.height, image.width)], device=device)
results = owlvit_processor.post_process_grounded_object_detection(
    outputs=owl_outputs,
    target_sizes=target_sizes,
    threshold=0.05,
    text_labels=text_labels,
)

result = results[0]
boxes = result["boxes"].detach().cpu()
scores = result["scores"].detach().cpu()
labels = result["text_labels"]

print("=" * 60)
print("OWL-ViT Detection: Base Categories")
print("=" * 60)
for box, score, label in zip(boxes, scores, labels):
    box_list = [round(v, 1) for v in box.tolist()]
    print(f"  {label:>25s}  score={score.item():.3f}  box={box_list}")

In [ ]:
# ── 场景2：加入更细粒度的新类别 query ──
text_labels_novel = [["a tabby cat", "a black cat", "a TV remote", "a couch", "a pink blanket"]]

owl_inputs_novel = owlvit_processor(text=text_labels_novel, images=image, return_tensors="pt")
owl_inputs_novel = {k: v.to(device) for k, v in owl_inputs_novel.items()}
with torch.no_grad():
    owl_outputs_novel = owlvit_model(**owl_inputs_novel)

results_novel = owlvit_processor.post_process_grounded_object_detection(
    outputs=owl_outputs_novel,
    target_sizes=torch.tensor([(image.height, image.width)], device=device),
    threshold=0.05,
    text_labels=text_labels_novel,
)

result_novel = results_novel[0]
boxes_n = result_novel["boxes"].detach().cpu()
scores_n = result_novel["scores"].detach().cpu()
labels_n = result_novel["text_labels"]

print("Open-vocabulary queries:")
for box, score, label in zip(boxes_n, scores_n, labels_n):
    box_list = [round(v, 1) for v in box.tolist()]
    print(f"  {label:>25s}  score={score.item():.3f}  box={box_list}")

In [ ]:
# ── 基础 query vs 新 query 可视化对比 ──
COLORS = ["#e74c3c", "#2ecc71", "#3498db", "#f39c12", "#9b59b6", "#1abc9c", "#e67e22", "#2c3e50"]


def draw_detections(ax, pil_image, boxes, scores, labels, title):
    ax.imshow(pil_image)
    label_to_color = {}
    color_idx = 0
    for box, score, label in zip(boxes, scores, labels):
        if label not in label_to_color:
            label_to_color[label] = COLORS[color_idx % len(COLORS)]
            color_idx += 1
        color = label_to_color[label]
        x1, y1, x2, y2 = box.tolist()
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(y1 - 5, 2), f"{label}: {score:.2f}", fontsize=8, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))
    ax.set_title(title, fontsize=11)
    ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
draw_detections(axes[0], image, boxes, scores, labels, "Base Queries")
draw_detections(axes[1], image, boxes_n, scores_n, labels_n, "Novel Queries")
plt.tight_layout()
plt.show()

## 4. 复试速记总结

### 4.1 把 ViLD 浓缩成 3 句话

1. **ViLD-text**：把 detector 的固定分类器替换成 `region embedding × text embedding`。
2. **ViLD-image**：用 CLIP image encoder 做蒸馏，让 detector 学到开放世界视觉语义。
3. **训练时有 image branch，推理时没有 image branch**：因为它只是 teacher，真正部署时只保留 detector + text encoder。

### 4.2 和后续工作的关系

| 方法 | 一句话理解 |
|------|-----------|
| ViLD | 用 CLIP teacher 蒸馏两阶段检测器 |
| Detic | 用大规模分类数据把 detector 的类别词表做大 |
| GLIP | 把 grounding 和 detection 更深地统一起来 |
| OWL-ViT | 直接从视觉语言预训练模型出发做文本条件检测 |

### 4.3 复试回答模板

如果老师问“ViLD 的创新点是什么”，最稳的回答是：

> ViLD 的关键不只是把 CLIP 文本特征拿来做分类，而是同时做了两件事：
> 第一，用文本 embedding 替代 closed-set 分类器，让 detector 支持任意类别文本查询；
> 第二，用 CLIP 图像编码器做 region-level distillation，把开放世界视觉语义蒸馏到 detector 中。
> 所以它既改了分类形式，也补了泛化能力。

## 最后只背这 5 问

**Q1：ViLD 为什么能识别训练时没见过的新类别？**

因为它不再用固定分类器，而是把 proposal 变成 `region embedding`，再和任意类别的 `text embedding` 做点积分类。

**Q2：ViLD-text 和 ViLD-image 分别做什么？**

- `ViLD-text`：把 closed-set 分类器改成文本驱动分类器
- `ViLD-image`：把 CLIP 的开放世界视觉语义蒸馏给 detector

**Q3：为什么 ViLD-image 训练时重要、推理时却不需要？**

因为它本质上是 teacher 分支。训练时负责教，推理时 student 已经学会了，就不需要 teacher 再参与。

**Q4：ViLD 和传统 Faster R-CNN 最大差别是什么？**

不是 proposal，而是第二阶段分类头：
- 传统 detector：输出固定类别 logits
- ViLD：输出 region embedding，再和文本 embedding 做相似度分类

**Q5：ViLD 的主要局限是什么？**

- 仍然依赖 proposal 质量
- CLIP 蒸馏成本高
- 工程链路比直接端到端方法更复杂